# Secure Data Orchestration – Case Study Implementation

This notebook evaluates the proposed Secure Data Orchestration framework using the MIMIC-III Clinical Database Demo (v1.4).

The objective is to examine predictive performance, computational feasibility, and clustering behavior within a structured Big Data Analytics workflow.

## Dataset and Pre-processing

The CHARTEVENTS table is utilized to extract continuous monitoring attributes:

- Heart Rate (ITEMID 211, 220045)
- Systolic Blood Pressure (ITEMID 220179)
- Diastolic Blood Pressure (ITEMID 220180)
- Oxygen Saturation (ITEMID 220277)

Data cleaning includes:
- Removal of null physiological measurements
- Aggregation of repeated measurements using mean values
- Mean imputation for incomplete vital sign attributes

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load dataset
df = pd.read_csv("CHARTEVENTS.csv", low_memory=False)
df.columns = df.columns.str.lower()

# Required ITEMIDs
vital_itemids = [211, 220045, 220179, 220180, 220277]

# Filter and remove null values
vitals = df[df["itemid"].isin(vital_itemids)]
vitals = vitals[["subject_id", "itemid", "valuenum"]].dropna()

# Mean aggregation per subject
vitals_pivot = vitals.pivot_table(
    index="subject_id",
    columns="itemid",
    values="valuenum",
    aggfunc="mean"
)

# Mean imputation
vitals_pivot = vitals_pivot.fillna(vitals_pivot.mean())

print("Number of patients retained:", len(vitals_pivot))

## Risk Label Definition

Risk = 1 if HR > 100 or SpO2 < 92  
Risk = 0 otherwise

In [ ]:
vitals_pivot["Risk"] = (
    (vitals_pivot[211] > 100) |
    (vitals_pivot[220277] < 92)
).astype(int)

## Experimental Setup

The dataset is partitioned into training and testing subsetsnusing a 70–30 split configuration for performance evaluation. Execution latency is measured during model training.

In [ ]:
features = [211, 220045, 220179, 220180, 220277]
X = vitals_pivot[features]
y = vitals_pivot["Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

start_time = time.time()

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

end_time = time.time()
execution_time = end_time - start_time

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("Execution Time (seconds):", execution_time)

## Confusion Matrix Analysis

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## ROC Analysis

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

print("AUC:", roc_auc)

## Clustering Analysis

K-Means clustering (k = 3) is performed to identify physiological groupings.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
vitals_pivot["Cluster"] = kmeans.fit_predict(X)

sns.scatterplot(
    x=vitals_pivot[211],
    y=vitals_pivot[220277],
    hue=vitals_pivot["Cluster"],
    palette="Set2"
)

plt.title("K-Means Clustering")
plt.show()